In [1]:
import pandas as pd
from pathlib import Path

In [2]:
data = "/home/jayo/repos/MIRROR-Eval/results/results_scored.jsonl"
df = pd.read_json(data, lines=True)

In [3]:
df.head()

,model_name,split_name,output_both_sets,output_set_1,output_set_2,prompt,src,set1,set2,set1_label,...,llm_diversity,correct_label,score_both_0,score_both_1,predicted_label_both,accuracy_both,score_set_1,score_set_2,predicted_label_individual,accuracy_individual_set
0,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(1, 4)",(1),(1),scale,We need bread on rainy days.,[Rainy days do not have any specific relation ...,[Rainy days don't have any correlation with th...,diversified,...,2,-1,1,4,1,0,1,1,-1,1
1,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(5, 4)",(1),(2),scale_with_examples,We need bread on rainy days.,[Rainy days do not have any specific relation ...,[Rainy days don't have any correlation with th...,diversified,...,2,-1,5,4,0,0,1,2,1,0
2,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(0, 1)",(0),(0),category,We need bread on rainy days.,[Rainy days do not have any specific relation ...,[Rainy days don't have any correlation with th...,diversified,...,2,-1,0,1,1,0,0,0,-1,1
3,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(0,1)",(0),(0),category_with_examples,We need bread on rainy days.,[Rainy days do not have any specific relation ...,[Rainy days don't have any correlation with th...,diversified,...,2,-1,0,1,1,0,0,0,-1,1
4,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(4, 1)",(3),(1),scale,She drove her car to space.,"[Space travel requires specialized vehicles, n...",[It is not currently possible for cars to trav...,icd,...,0,0,4,1,0,1,3,1,0,1


In [4]:
# correct label is -1, 0, or 1.
# Check the distribution of each label per dataset
# That means consider only one model and prompt
df_filtered = df[(df['model_name'] == 'meta-llama/Llama-3.3-70B-Instruct') & (df['prompt'] == 'scale')]

In [7]:
df_filtered.head()

,model_name,split_name,output_both_sets,output_set_1,output_set_2,prompt,src,set1,set2,set1_label,...,llm_diversity,correct_label,score_both_0,score_both_1,predicted_label_both,accuracy_both,score_set_1,score_set_2,predicted_label_individual,accuracy_individual_set
0,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(1, 4)",(1),(1),scale,We need bread on rainy days.,[Rainy days do not have any specific relation ...,[Rainy days don't have any correlation with th...,diversified,...,2,-1,1,4,1,0,1,1,-1,1
4,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(4, 1)",(3),(1),scale,She drove her car to space.,"[Space travel requires specialized vehicles, n...",[It is not currently possible for cars to trav...,icd,...,0,0,4,1,0,1,3,1,0,1
8,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(1, 1)",(4),(2),scale,He deposited drugs into the bank.,[Drug deposits are a criminal offense and woul...,[Banks are not meant for storing illegal subst...,icd,...,2,-1,1,1,-1,1,4,2,0,0
12,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(1, 4)",(1),(1),scale,I'm dirty now so I need to have lunch.,[Being dirty does not necessarily have any rel...,[The level of cleanliness does not determine o...,default,...,1,1,1,4,1,1,1,1,-1,0
16,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(2, 4)",(4),(2),scale,Programmers usually don't use computers.,[Programmers need computers to write and test ...,[Programmers need computers to write and test ...,default,...,1,1,2,4,1,1,4,2,0,0


In [ ]:
df_filtered['correct_label'].value_counts() #1920
# So 1288 of the first sets are more diverse, 553 of the second sets, and 79 are equally diverse.

correct_label
 0    1288
 1     553
-1      79
Name: count, dtype: int64

In [11]:
# There are three datasets, do value counts for each dataset
for dataset in df_filtered['split_name'].unique():
    print(f"Dataset: {dataset}")
    subset = df_filtered[df_filtered['split_name'] == dataset]
    print(subset['correct_label'].value_counts())
    # print total number of samples
    print(f"Total samples: {len(subset)}")
    print()

Dataset: semeval_evaluation
correct_label
 0    378
 1    238
-1     79
Name: count, dtype: int64
Total samples: 695

Dataset: dimongen_evaluation
correct_label
0    468
1    235
Name: count, dtype: int64
Total samples: 703

Dataset: commongen_evaluation
correct_label
0    442
1     80
Name: count, dtype: int64
Total samples: 522



In [5]:
# Create a new empty dataframe
# We want to collect the accuracy and confidence intervals for each model/prompt/dataset combination
df_summary = pd.DataFrame()

In [6]:
from scipy.stats import binomtest

In [8]:
# Iterate over each unique combination of model_name, prompt, and dataset
for (model_name, prompt, split_name), group in df.groupby(['model_name', 'prompt', 'split_name']):
    total = len(group)
    correct_both = group['accuracy_both'].sum()
    correct_individual = group['accuracy_individual_set'].sum()

    accuracy_both = correct_both / total if total > 0 else 0
    accuracy_individual = correct_individual / total if total > 0 else 0

    result_both = binomtest(correct_both, total)
    result_individual = binomtest(correct_individual, total)

    both_ci_low, both_ci_high = result_both.proportion_ci(confidence_level=0.95, method="exact")
    individual_ci_low, individual_ci_high = result_individual.proportion_ci(confidence_level=0.95, method="exact")

    # Append the results to the summary dataframe
    df_summary = pd.concat([df_summary, pd.DataFrame({
        'model_name': [model_name],
        'prompt': [prompt],
        'split_name': [split_name],
        'accuracy_both': [accuracy_both],
        'accuracy_individual': [accuracy_individual],
        'ci_both_low': [both_ci_low],
        'ci_both_high': [both_ci_high],
        'ci_individual_low': [individual_ci_low],
        'ci_individual_high': [individual_ci_high],
    })], ignore_index=True)

In [10]:
df_summary

,model_name,prompt,split_name,accuracy_both,accuracy_individual,ci_both_low,ci_both_high,ci_individual_low,ci_individual_high
0,Qwen/Qwen3-30B-A3B-Instruct-2507,category,commongen_evaluation,0.178161,0.057471,0.146277,0.213739,0.039108,0.081030
1,Qwen/Qwen3-30B-A3B-Instruct-2507,category,dimongen_evaluation,0.359886,0.231863,0.324347,0.396607,0.201135,0.264861
2,Qwen/Qwen3-30B-A3B-Instruct-2507,category,semeval_evaluation,0.366906,0.146763,0.330986,0.403962,0.121279,0.175287
3,Qwen/Qwen3-30B-A3B-Instruct-2507,category_with_examples,commongen_evaluation,0.214559,0.101533,0.180085,0.252302,0.076983,0.130700
4,Qwen/Qwen3-30B-A3B-Instruct-2507,category_with_examples,dimongen_evaluation,0.364154,0.268848,0.328506,0.400947,0.236391,0.303259
5,Qwen/Qwen3-30B-A3B-Instruct-2507,category_with_examples,semeval_evaluation,0.352518,0.152518,0.316970,0.389324,0.126591,0.181435
6,Qwen/Qwen3-30B-A3B-Instruct-2507,scale,commongen_evaluation,0.409962,0.318008,0.367426,0.453522,0.278227,0.359864
7,Qwen/Qwen3-30B-A3B-Instruct-2507,scale,dimongen_evaluation,0.574680,0.435277,0.537173,0.611558,0.398254,0.472846
8,Qwen/Qwen3-30B-A3B-Instruct-2507,scale,semeval_evaluation,0.471942,0.341007,0.434296,0.509828,0.305783,0.377588
9,Qwen/Qwen3-30B-A3B-Instruct-2507,scale_with_examples,commongen_evaluation,0.649425,0.392720,0.606779,0.690369,0.350579,0.436083


In [35]:
chamfer_cossim_data = "/home/jayo/repos/MIRROR-Eval/results/chamfer_cossim.jsonl"

In [36]:
df = pd.read_json(chamfer_cossim_data, lines=True)

In [26]:
df.head()

,model_name,split_name,output_both_sets,output_set_1,output_set_2,prompt,src,set1,set2,set1_label,...,Diversity_Set1,Diversity_Set2,llm_quality,llm_diversity,chamfer_distance_set1,chamfer_distance_set2,chamfer_distance_output,self_avgcosine_set1,self_avgcosine_set2,self_avgcosine_output
0,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(1, 4)",(1),(1),scale,We need bread on rainy days.,[Rainy days do not have any specific relation ...,[Rainy days don't have any correlation with th...,diversified,...,3.0,3.0,0,2,0.000750,0.008349,1,0.998719,0.983025,1
1,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(4, 1)",(3),(1),scale,She drove her car to space.,"[Space travel requires specialized vehicles, n...",[It is not currently possible for cars to trav...,icd,...,4.4,2.2,0,0,0.005265,0.000116,0,0.991856,0.999880,0
2,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(1, 1)",(4),(2),scale,He deposited drugs into the bank.,[Drug deposits are a criminal offense and woul...,[Banks are not meant for storing illegal subst...,icd,...,3.0,3.0,2,2,0.006236,0.004494,0,0.991253,0.993072,0
3,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(1, 4)",(1),(1),scale,I'm dirty now so I need to have lunch.,[Being dirty does not necessarily have any rel...,[The level of cleanliness does not determine o...,default,...,2.4,3.6,2,1,0.000731,0.000905,1,0.998778,0.998589,1
4,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(2, 4)",(4),(2),scale,Programmers usually don't use computers.,[Programmers need computers to write and test ...,[Programmers need computers to write and test ...,default,...,2.6,3.0,1,1,0.004927,0.007769,1,0.990588,0.987567,1


In [28]:
df["correct_label"] = (df["Diversity_Set1"] < df["Diversity_Set2"]).astype(int)
ties = df["Diversity_Set1"] == df["Diversity_Set2"]
df.loc[ties, "correct_label"] = -1

df["chamfer_accuracy"] = (df["correct_label"] == df["chamfer_distance_output"]).astype(int)
df["cossim_accuracy"] = (df["correct_label"] == df["self_avgcosine_output"]).astype(int)

In [29]:
# print the first 50 rows
df.head()

,model_name,split_name,output_both_sets,output_set_1,output_set_2,prompt,src,set1,set2,set1_label,...,llm_diversity,chamfer_distance_set1,chamfer_distance_set2,chamfer_distance_output,self_avgcosine_set1,self_avgcosine_set2,self_avgcosine_output,correct_label,chamfer_accuracy,cossim_accuracy
0,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(1, 4)",(1),(1),scale,We need bread on rainy days.,[Rainy days do not have any specific relation ...,[Rainy days don't have any correlation with th...,diversified,...,2,0.000750,0.008349,1,0.998719,0.983025,1,-1,0,0
1,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(4, 1)",(3),(1),scale,She drove her car to space.,"[Space travel requires specialized vehicles, n...",[It is not currently possible for cars to trav...,icd,...,0,0.005265,0.000116,0,0.991856,0.999880,0,0,1,1
2,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(1, 1)",(4),(2),scale,He deposited drugs into the bank.,[Drug deposits are a criminal offense and woul...,[Banks are not meant for storing illegal subst...,icd,...,2,0.006236,0.004494,0,0.991253,0.993072,0,-1,0,0
3,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(1, 4)",(1),(1),scale,I'm dirty now so I need to have lunch.,[Being dirty does not necessarily have any rel...,[The level of cleanliness does not determine o...,default,...,1,0.000731,0.000905,1,0.998778,0.998589,1,1,1,1
4,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(2, 4)",(4),(2),scale,Programmers usually don't use computers.,[Programmers need computers to write and test ...,[Programmers need computers to write and test ...,default,...,1,0.004927,0.007769,1,0.990588,0.987567,1,1,1,1


In [32]:
df_summary = pd.DataFrame()
# Iterate over each unique combination of model_name, and dataset
for split_name, group in df.groupby(['split_name']):
    total = len(group)
    correct_chamfer = group['chamfer_accuracy'].sum()
    correct_cossim = group['cossim_accuracy'].sum()

    accuracy_chamfer = correct_chamfer / total if total > 0 else 0
    accuracy_cossim = correct_cossim / total if total > 0 else 0

    result_chamfer = binomtest(correct_chamfer, total)
    result_cossim = binomtest(correct_cossim, total)

    chamfer_ci_low, chamfer_ci_high = result_chamfer.proportion_ci(confidence_level=0.95, method="exact")
    cossim_ci_low, cossim_ci_high = result_cossim.proportion_ci(confidence_level=0.95, method="exact")
    # Append the results to the summary dataframe
    df_summary = pd.concat([df_summary, pd.DataFrame({
        'split_name': [split_name],
        'accuracy_chamfer': [accuracy_chamfer],
        'accuracy_cossim': [accuracy_cossim],
        'ci_chamfer_low': [chamfer_ci_low],
        'ci_chamfer_high': [chamfer_ci_high],
        'ci_cossim_low': [cossim_ci_low],
        'ci_cossim_high': [cossim_ci_high],
    })], ignore_index=True)

In [33]:
df_summary

,split_name,accuracy_chamfer,accuracy_cossim,ci_chamfer_low,ci_chamfer_high,ci_cossim_low,ci_cossim_high
0,"(commongen_evaluation,)",0.559387,0.528736,0.534387,0.584164,0.503659,0.553704
1,"(dimongen_evaluation,)",0.504979,0.485064,0.483409,0.526534,0.463531,0.506639
2,"(semeval_evaluation,)",0.528058,0.507914,0.506363,0.549673,0.486216,0.529589
